# Module 07-3: LLM 응답 캐싱

## 학습 목표
- `compute_cache_key()` 해시 함수로 LLM 입력을 결정론적으로 식별할 수 있다
- TTL(Time To Live) 기반 `LLMCache`를 구현하여 중복 호출을 방지할 수 있다
- `hit_rate` 측정으로 캐싱 효과를 정량화할 수 있다
- 캐싱이 적합한 태스크와 부적합한 태스크를 구분할 수 있다

**참조 문서**: `docs/part3-prompt-and-llm/07-llm-call-optimization.md` 섹션 2.3

## 개념 요약

### 캐싱 적합도

| 상황 | 캐싱 적합도 | 이유 |
|------|-----------|------|
| 동일 코드를 여러 번 분석 | HIGH | 같은 입력 = 같은 결과 |
| 재처리/재시도 | HIGH | 이전 결과 재사용 가능 |
| 코드 생성 (비결정적) | LOW | 매번 다른 결과가 나올 수 있음 |
| 사용자 대화 | LOW | 맥락이 매번 다름 |

### 캐시 키 설계 원칙

```python
# 입력 딕셔너리의 키 순서가 달라도 같은 해시값을 생성해야 합니다
key1 = compute_cache_key("analyze", "sonnet", {"code": "def f(): pass", "lang": "python"})
key2 = compute_cache_key("analyze", "sonnet", {"lang": "python", "code": "def f(): pass"})
assert key1 == key2  # sort_keys=True로 정렬된 JSON 직렬화 후 해시
```

### TTL 기반 캐시 동작
```
캐시 조회 → 키 존재? → Yes → 만료? → No → HIT (반환)
                          ↓ Yes     ↓ Yes
                       MISS      만료 항목 삭제 → MISS
```

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

import hashlib
import json
import time
import logging
from typing import Any
from common.fake_llm import FakeLLM

logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(__name__)

print("환경 설정 완료")

## 실습 1: compute_cache_key() 해시 함수 구현

동일한 태스크/모델/입력이면 항상 같은 해시값을 반환하는 함수를 구현하세요.

**요구사항**:
- task_name, model, inputs 딕셔너리를 받아 SHA-256 해시값(hex 문자열) 반환
- 입력 딕셔너리의 키 순서가 달라도 동일한 해시값 생성
- `json.dumps(..., sort_keys=True, ensure_ascii=False)` 활용

In [ ]:
# TODO: 여기에 코드를 작성하세요
#
# 힌트 1 (방향): key_data 딕셔너리를 JSON으로 직렬화한 후 SHA-256 해시를 계산합니다.
#               sort_keys=True로 키 순서를 정규화합니다.
#
# 힌트 2 (키워드): hashlib.sha256, json.dumps(sort_keys=True), .encode("utf-8"), .hexdigest()
#
# 힌트 3 (거의 정답):
#   def compute_cache_key(task_name: str, model: str, inputs: dict) -> str:
#       key_data = {"task": task_name, "model": model, "inputs": inputs}
#       serialized = json.dumps(key_data, sort_keys=True, ensure_ascii=False)
#       return hashlib.sha256(serialized.encode("utf-8")).hexdigest()

def compute_cache_key(task_name: str, model: str, inputs: dict) -> str:
    """LLM 호출의 캐시 키를 생성한다.
    
    Args:
        task_name: 태스크 이름
        model: 모델 이름
        inputs: 입력 데이터 딕셔너리
    
    Returns:
        SHA-256 해시값 (64자 hex 문자열)
    """
    pass  # TODO: 구현하세요


print("compute_cache_key 함수 정의 완료")

In [ ]:
# 검증 셀
# 같은 입력 → 같은 키
key1 = compute_cache_key("analyze", "sonnet", {"code": "def f(): pass", "lang": "python"})
key2 = compute_cache_key("analyze", "sonnet", {"lang": "python", "code": "def f(): pass"})
assert key1 == key2, f"키 순서가 달라도 같은 해시여야 합니다. key1={key1[:16]}..., key2={key2[:16]}..."

# 다른 입력 → 다른 키
key3 = compute_cache_key("analyze", "sonnet", {"code": "def g(): return 1"})
assert key1 != key3, "다른 입력은 다른 해시여야 합니다"

# 다른 태스크 → 다른 키
key4 = compute_cache_key("summarize", "sonnet", {"code": "def f(): pass", "lang": "python"})
assert key1 != key4, "다른 태스크는 다른 해시여야 합니다"

# 키 길이 확인 (SHA-256 hex = 64자)
assert len(key1) == 64, f"SHA-256 해시는 64자여야 합니다. 현재: {len(key1)}자"

print(f"키 예시: {key1[:32]}...")
print("실습 1 완료! 키 순서 독립성 확인")

## 실습 2: LLMCache 클래스 구현

TTL 기반 인메모리 캐시 `LLMCache`를 구현하세요.

**`__init__`**: default_ttl, max_entries, _store, _hits, _misses 초기화

**`get(task, model, inputs)`**:
- 캐시 키 계산
- 키 존재하면 만료 여부 확인
- 만료 안 됐으면 _hits 증가 후 반환
- 만료됐거나 없으면 _misses 증가 후 None 반환

**`set(task, model, inputs, result, ttl=None)`**:
- 용량 초과 시 가장 오래된 항목 제거
- (task, model, inputs) 키로 (result, expires_at) 저장

**`hit_rate` 프로퍼티**: hits / (hits + misses)

In [ ]:
# TODO: 여기에 코드를 작성하세요
#
# 힌트 1 (방향): _store는 {key: (value, expires_at)} 형식의 딕셔너리입니다.
#               time.time() + ttl로 만료 시간을 계산합니다.
#
# 힌트 2 (키워드): compute_cache_key(), time.time() < expires_at, del self._store[key]
#               min(self._store, key=lambda k: self._store[k][1])
#
# 힌트 3 (거의 정답):
#   def get(self, task, model, inputs):
#       key = compute_cache_key(task, model, inputs)
#       if key in self._store:
#           value, expires_at = self._store[key]
#           if time.time() < expires_at:
#               self._hits += 1
#               return value
#           else:
#               del self._store[key]
#       self._misses += 1
#       return None

class LLMCache:
    """LLM 응답 캐시 (메모리 기반, TTL 지원)."""

    def __init__(self, default_ttl: int = 3600, max_entries: int = 500):
        self._default_ttl = default_ttl
        self._max_entries = max_entries
        self._store: dict[str, tuple[Any, float]] = {}
        self._hits = 0
        self._misses = 0

    def get(self, task: str, model: str, inputs: dict) -> Any | None:
        """캐시에서 결과를 조회한다. 없거나 만료되면 None."""
        # TODO: 구현하세요
        pass

    def set(self, task: str, model: str, inputs: dict, result: Any, ttl: int | None = None):
        """결과를 캐시에 저장한다."""
        # TODO: 구현하세요
        pass

    @property
    def hit_rate(self) -> float:
        """캐시 히트율 (0.0 ~ 1.0)."""
        # TODO: 구현하세요
        pass

    def get_stats(self) -> dict:
        """캐시 통계를 반환한다."""
        return {
            "hits": self._hits,
            "misses": self._misses,
            "hit_rate": round(self.hit_rate * 100, 1),
            "entries": len(self._store),
        }


print("LLMCache 클래스 정의 완료")

In [ ]:
# 검증 셀: 기본 동작
cache = LLMCache(default_ttl=60)
inputs = {"code": "def hello(): pass", "lang": "python"}

# 처음 조회: MISS
result = cache.get("analyze", "sonnet", inputs)
assert result is None, "처음 조회는 None이어야 합니다 (MISS)"
assert cache._misses == 1, f"misses가 1이어야 합니다. 현재: {cache._misses}"

# 저장
cache.set("analyze", "sonnet", inputs, {"bugs": [], "score": 9})
assert len(cache._store) == 1, f"저장 후 항목 수가 1이어야 합니다. 현재: {len(cache._store)}"

# 다시 조회: HIT
result = cache.get("analyze", "sonnet", inputs)
assert result is not None, "저장 후 조회는 값을 반환해야 합니다 (HIT)"
assert result == {"bugs": [], "score": 9}, f"반환값이 다릅니다. 현재: {result}"
assert cache._hits == 1, f"hits가 1이어야 합니다. 현재: {cache._hits}"

# hit_rate 확인: 1 hit / (1 hit + 1 miss) = 0.5
assert abs(cache.hit_rate - 0.5) < 0.001, f"hit_rate가 0.5여야 합니다. 현재: {cache.hit_rate}"

print(f"통계: {cache.get_stats()}")
print("기본 동작 검증 완료!")

In [ ]:
# 검증 셀: TTL 만료
cache_short = LLMCache(default_ttl=1)  # 1초 TTL
cache_short.set("test", "haiku", {"input": "x"}, "result_value")

# 즉시 조회: HIT
r1 = cache_short.get("test", "haiku", {"input": "x"})
assert r1 == "result_value", f"즉시 조회는 값을 반환해야 합니다. 현재: {r1}"

# 2초 대기 후 조회: MISS (만료)
time.sleep(2)
r2 = cache_short.get("test", "haiku", {"input": "x"})
assert r2 is None, f"만료 후 조회는 None이어야 합니다. 현재: {r2}"

print("TTL 만료 검증 완료!")
print("실습 2 완료!")

## 실습 3: hit_rate 측정 시나리오

FakeLLM과 LLMCache를 연동하여 캐싱 효과를 측정하세요.

**시나리오**: 10개의 코드 분석 요청 중 4개는 이미 캐시된 데이터와 동일
- 캐시 없이: 10번 LLM 호출
- 캐시 있으면: 약 40% hit rate

In [ ]:
# TODO: 여기에 코드를 작성하세요
#
# 힌트 1 (방향): FakeLLM을 사용하여 LLM 호출을 시뮬레이션합니다.
#               캐시 미스 시에만 FakeLLM을 호출하고 결과를 캐시에 저장합니다.
#
# 힌트 2 (키워드): cached = cache.get(task, model, inputs)
#                if cached is None: result = llm.invoke(...); cache.set(...)
#
# 힌트 3 (거의 정답):
#   llm_call_count = 0
#   def cached_analyze(code: str, task: str = "analyze", model: str = "haiku") -> str:
#       global llm_call_count
#       inputs = {"code": code}
#       cached = cache.get(task, model, inputs)
#       if cached is not None:
#           return cached
#       llm_call_count += 1
#       result = llm.invoke(f"코드를 분석해주세요: {code}").content
#       cache.set(task, model, inputs, result)
#       return result

cache = LLMCache(default_ttl=300)
llm = FakeLLM(responses={"분석": "분석 완료: 버그 없음"}, default_response="처리 완료")
llm_call_count = 0

def cached_analyze(code: str, task: str = "analyze", model: str = "haiku") -> str:
    """캐싱을 적용한 코드 분석 함수."""
    global llm_call_count
    # TODO: 캐시 조회 → 미스 시 LLM 호출 → 결과 캐시 저장 → 반환
    pass  # 이 줄을 지우고 구현하세요


print("cached_analyze 함수 정의 완료")

In [ ]:
# 10개 요청으로 히트율 테스트
# 코드 A: 4번 요청 (1번 미스, 3번 히트)
# 코드 B: 3번 요청 (1번 미스, 2번 히트)
# 코드 C: 2번 요청 (1번 미스, 1번 히트)
# 코드 D: 1번 요청 (1번 미스)
# 총: 10번 요청, 4번 미스, 6번 히트 → 60% hit rate

requests = [
    "def add(a, b): return a + b",   # 코드 A - 1번째
    "def multiply(a, b): return a * b",  # 코드 B - 1번째
    "def add(a, b): return a + b",   # 코드 A - 2번째 (HIT)
    "def greet(name): return f'Hello {name}'",  # 코드 C - 1번째
    "def multiply(a, b): return a * b",  # 코드 B - 2번째 (HIT)
    "def square(x): return x ** 2",  # 코드 D - 1번째
    "def add(a, b): return a + b",   # 코드 A - 3번째 (HIT)
    "def greet(name): return f'Hello {name}'",  # 코드 C - 2번째 (HIT)
    "def multiply(a, b): return a * b",  # 코드 B - 3번째 (HIT)
    "def add(a, b): return a + b",   # 코드 A - 4번째 (HIT)
]

print("=" * 50)
print("LLM 호출 시뮬레이션 시작")
print("=" * 50)

results = []
for i, code in enumerate(requests, 1):
    result = cached_analyze(code)
    results.append(result)
    status = "MISS" if cache._hits + cache._misses - (i-1) <= cache._misses else "HIT"
    print(f"요청 {i:2d}: {code[:30]:<32} → {result[:20]}")

print()
stats = cache.get_stats()
print(f"총 요청: {len(requests)}개")
print(f"LLM 실제 호출: {llm_call_count}회")
print(f"캐시 히트: {stats['hits']}회 / 미스: {stats['misses']}회")
print(f"히트율: {stats['hit_rate']}%")

In [ ]:
# 검증 셀
assert llm_call_count is not None, "llm_call_count가 정의되어야 합니다"
assert cache._hits > 0, "캐시 히트가 발생해야 합니다"
assert cache._misses > 0, "캐시 미스가 발생해야 합니다"
assert cache.hit_rate > 0, "hit_rate가 0보다 커야 합니다"

# 10번 요청에서 4번의 고유 코드 → LLM 4번 호출, 6번 캐시 히트
assert llm_call_count == 4, f"LLM은 4번만 호출되어야 합니다 (고유 코드 4개). 현재: {llm_call_count}번"
assert cache._hits == 6, f"캐시 히트가 6번이어야 합니다. 현재: {cache._hits}번"
assert abs(cache.hit_rate - 0.6) < 0.01, f"hit_rate가 60%여야 합니다. 현재: {cache.hit_rate:.1%}"

print(f"✅ 실습 3 완료! hit_rate={cache.hit_rate:.1%} (LLM 호출 {10 - llm_call_count}번 절약)")

## 실습 4: max_entries 용량 관리

`max_entries`를 초과할 때 가장 오래된 항목이 제거되는지 확인하세요.

In [ ]:
# 용량 제한 테스트
small_cache = LLMCache(default_ttl=300, max_entries=3)

# 3개 항목 추가
small_cache.set("t1", "m", {"i": "1"}, "result_1")
time.sleep(0.01)  # 타임스탬프 차이를 위해 짧게 대기
small_cache.set("t2", "m", {"i": "2"}, "result_2")
time.sleep(0.01)
small_cache.set("t3", "m", {"i": "3"}, "result_3")

assert len(small_cache._store) == 3, f"3개 항목이 있어야 합니다. 현재: {len(small_cache._store)}"

# 4번째 항목 추가 → 가장 오래된 항목(t1) 제거
small_cache.set("t4", "m", {"i": "4"}, "result_4")
assert len(small_cache._store) == 3, f"max_entries=3 초과 후에도 3개여야 합니다. 현재: {len(small_cache._store)}"

# t1은 제거됨, t4는 존재
r1 = small_cache.get("t1", "m", {"i": "1"})
r4 = small_cache.get("t4", "m", {"i": "4"})

assert r1 is None, "가장 오래된 항목(t1)이 제거되어야 합니다"
assert r4 == "result_4", f"새로 추가된 t4는 존재해야 합니다. 현재: {r4}"

print(f"용량 관리 테스트 통과: max_entries={small_cache._max_entries}, 현재 항목={len(small_cache._store)}")
print("실습 4 완료!")

## 정리

이번 실습에서 학습한 내용:

1. **compute_cache_key()**: SHA-256 해시로 결정론적 캐시 키 생성 (키 순서 독립성)
2. **LLMCache**: TTL 기반 인메모리 캐시로 중복 LLM 호출 방지
3. **hit_rate 측정**: 캐싱 효과를 정량화하여 최적화 효과 확인
4. **max_entries 관리**: 메모리 사용량 제한으로 운영 안정성 확보

**캐싱 효과**: 10번 요청 중 6번 캐시 히트 → LLM 호출 60% 감소, 비용과 지연 시간 절감

**다음 모듈 안내**: **Module 08: 에러 처리** - 재시도, Circuit Breaker, 복원력 있는 에이전트 구현